#Differential Expression with DESeq2
   RNA-Sequence Analysis Workflow

1. Install packages and load libraries
2. Read gene counts and meta data into a data frame
3. Quality assess and clean raw sequencing data
4. Align reads to a reference
5. Count the number of reads assigned to each contig/gene
6. Extract counts and store in a matrix
7. Create column metadata table
8. Analyze count data using DESEQ2

`Sk. Tanzir Mehedi`

`Lecturer, Department of IT, UITS`


Install package `Biobase`

In [1]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("Biobase")

Bioconductor version 3.12 (BiocManager 1.30.10), R 4.0.4 (2021-02-15)

Installing package(s) 'Biobase'



package 'Biobase' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\shuvo\AppData\Local\Temp\RtmpENgkzd\downloaded_packages


Installation path not writeable, unable to update packages: boot, cluster,
  MASS, mgcv, survival

Old packages: 'BiocManager', 'broom', 'caTools', 'cli', 'cpp11',
  'DelayedArray', 'RCurl', 'RSQLite', 'tinytex', 'utf8', 'vctrs'



Install package `DESeq2`

In [2]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("DESeq2")

Bioconductor version 3.12 (BiocManager 1.30.10), R 4.0.4 (2021-02-15)

Installing package(s) 'DESeq2'



package 'DESeq2' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\shuvo\AppData\Local\Temp\RtmpENgkzd\downloaded_packages


Installation path not writeable, unable to update packages: boot, cluster,
  MASS, mgcv, survival

Old packages: 'BiocManager', 'broom', 'caTools', 'cli', 'cpp11',
  'DelayedArray', 'RCurl', 'RSQLite', 'tinytex', 'utf8', 'vctrs'



Load libraries

In [34]:
library(DESeq2)
library(ggplot2)
library(dplyr)
library(readr)
library(RColorBrewer)

Read gene counts data

In [35]:
countData <- read.csv('GSE147507_filtered_countdata.csv', header = TRUE, sep = ",")
head(countData)

,X,GSM4432378,GSM4432379,GSM4432380,GSM4432381,GSM4432382,GSM4432383,GSM4432384,GSM4432385,GSM4432386,...,GSM4462375,GSM4462376,GSM4462377,GSM4462378,GSM4462379,GSM4462380,GSM4462413,GSM4462414,GSM4462415,GSM4462416
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,TSPAN6,14.62543,18.09892,16.68145,15.43346,11.59576,14.28377,52.38017,48.65381,41.60755,...,25.49182,27.37043,24.89059,28.09773,26.78680,21.74494,6.65122,19.66002,0.00000,2.62703
2,TNMD,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.02682,0.06676,0.00000,0.00000
3,DPM1,49.05343,51.70145,67.22577,51.20106,52.43619,44.82063,41.49314,43.33602,37.36188,...,56.26226,57.47108,59.41039,56.58570,50.34745,51.91360,58.46637,22.16342,0.00000,6.56757
4,SCYL3,11.97214,11.61063,11.72701,10.31336,11.52283,9.52251,11.14937,10.71495,12.31244,...,9.26975,10.51134,9.62921,9.26835,10.16725,8.03192,7.32171,9.57967,19.44372,12.47838
5,C1orf112,4.20643,4.02957,4.09082,4.02294,4.01111,4.61169,26.75848,23.73163,25.36787,...,6.43733,9.00972,5.81386,6.24394,7.82096,5.77906,2.25283,1.40190,9.72186,0.65676
6,FGR,0.25886,0.06830,0.04545,0.14629,0.07293,0.14957,0.00000,0.00000,0.00000,...,0.06437,0.00000,0.00000,0.00000,0.09776,0.09795,62.73066,105.47652,836.07976,1223.53763


Check row and colums of gene counts data

In [36]:
ncol(countData)
nrow(countData)

[1] 70

[1] 28125

Transform gene counts data into data frame

In [37]:
countDataFrame <- data.frame(countData)
head(countDataFrame)

,X,GSM4432378,GSM4432379,GSM4432380,GSM4432381,GSM4432382,GSM4432383,GSM4432384,GSM4432385,GSM4432386,...,GSM4462375,GSM4462376,GSM4462377,GSM4462378,GSM4462379,GSM4462380,GSM4462413,GSM4462414,GSM4462415,GSM4462416
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,TSPAN6,14.62543,18.09892,16.68145,15.43346,11.59576,14.28377,52.38017,48.65381,41.60755,...,25.49182,27.37043,24.89059,28.09773,26.78680,21.74494,6.65122,19.66002,0.00000,2.62703
2,TNMD,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.02682,0.06676,0.00000,0.00000
3,DPM1,49.05343,51.70145,67.22577,51.20106,52.43619,44.82063,41.49314,43.33602,37.36188,...,56.26226,57.47108,59.41039,56.58570,50.34745,51.91360,58.46637,22.16342,0.00000,6.56757
4,SCYL3,11.97214,11.61063,11.72701,10.31336,11.52283,9.52251,11.14937,10.71495,12.31244,...,9.26975,10.51134,9.62921,9.26835,10.16725,8.03192,7.32171,9.57967,19.44372,12.47838
5,C1orf112,4.20643,4.02957,4.09082,4.02294,4.01111,4.61169,26.75848,23.73163,25.36787,...,6.43733,9.00972,5.81386,6.24394,7.82096,5.77906,2.25283,1.40190,9.72186,0.65676
6,FGR,0.25886,0.06830,0.04545,0.14629,0.07293,0.14957,0.00000,0.00000,0.00000,...,0.06437,0.00000,0.00000,0.00000,0.09776,0.09795,62.73066,105.47652,836.07976,1223.53763


Transform gene counts data into integer for processing

In [38]:
countDataFrameRound <-countDataFrame %>% mutate(across(where(is.numeric), round, 3))
head(countDataFrameRound)

,X,GSM4432378,GSM4432379,GSM4432380,GSM4432381,GSM4432382,GSM4432383,GSM4432384,GSM4432385,GSM4432386,...,GSM4462375,GSM4462376,GSM4462377,GSM4462378,GSM4462379,GSM4462380,GSM4462413,GSM4462414,GSM4462415,GSM4462416
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,TSPAN6,14.625,18.099,16.681,15.433,11.596,14.284,52.380,48.654,41.608,...,25.492,27.370,24.891,28.098,26.787,21.745,6.651,19.660,0.000,2.627
2,TNMD,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.027,0.067,0.000,0.000
3,DPM1,49.053,51.701,67.226,51.201,52.436,44.821,41.493,43.336,37.362,...,56.262,57.471,59.410,56.586,50.347,51.914,58.466,22.163,0.000,6.568
4,SCYL3,11.972,11.611,11.727,10.313,11.523,9.523,11.149,10.715,12.312,...,9.270,10.511,9.629,9.268,10.167,8.032,7.322,9.580,19.444,12.478
5,C1orf112,4.206,4.030,4.091,4.023,4.011,4.612,26.758,23.732,25.368,...,6.437,9.010,5.814,6.244,7.821,5.779,2.253,1.402,9.722,0.657
6,FGR,0.259,0.068,0.045,0.146,0.073,0.150,0.000,0.000,0.000,...,0.064,0.000,0.000,0.000,0.098,0.098,62.731,105.477,836.080,1223.538


Read meta data

In [39]:
metaData <- read.csv('GSE147507_filtered_metadata.csv', header = TRUE, sep = ",")
head(metaData)

,X,treatment
,<chr>,<chr>
1,GSM4432378,Mock treated NHBE cells
2,GSM4432379,Mock treated NHBE cells
3,GSM4432380,Mock treated NHBE cells
4,GSM4432381,SARS-CoV-2 infected NHBE cells
5,GSM4432382,SARS-CoV-2 infected NHBE cells
6,GSM4432383,SARS-CoV-2 infected NHBE cells


Check row and colums of meta data

In [40]:
colnames(metaData)
ncol(metaData)
nrow(metaData)

[1] "X"         "treatment"

[1] 2

[1] 69

Transform meta data into data frame

In [41]:
metaDataFrame <- data.frame(metaData, row.names=1)
head(metaDataFrame)

,treatment
,<chr>
GSM4432378,Mock treated NHBE cells
GSM4432379,Mock treated NHBE cells
GSM4432380,Mock treated NHBE cells
GSM4432381,SARS-CoV-2 infected NHBE cells
GSM4432382,SARS-CoV-2 infected NHBE cells
GSM4432383,SARS-CoV-2 infected NHBE cells


Check gene counts and meta data dimension

In [42]:
rownames(metaDataFrame)
colnames(countDataFrameRound)

[1] "GSM4432378" "GSM4432379" "GSM4432380" "GSM4432381" "GSM4432382"
 [6] "GSM4432383" "GSM4432384" "GSM4432385" "GSM4432386" "GSM4432387"
[11] "GSM4432388" "GSM4432389" "GSM4432390" "GSM4432391" "GSM4432392"
[16] "GSM4432393" "GSM4432394" "GSM4432395" "GSM4432396" "GSM4432397"
[21] "GSM4462336" "GSM4462337" "GSM4462338" "GSM4462339" "GSM4462340"
[26] "GSM4462341" "GSM4462342" "GSM4462343" "GSM4462344" "GSM4462345"
[31] "GSM4462346" "GSM4462347" "GSM4462348" "GSM4462349" "GSM4462350"
[36] "GSM4462351" "GSM4462352" "GSM4462353" "GSM4462354" "GSM4462355"
[41] "GSM4462356" "GSM4462357" "GSM4462358" "GSM4462359" "GSM4462360"
[46] "GSM4462361" "GSM4462362" "GSM4462363" "GSM4462364" "GSM4462365"
[51] "GSM4462366" "GSM4462367" "GSM4462368" "GSM4462369" "GSM4462370"
[56] "GSM4462371" "GSM4462372" "GSM4462373" "GSM4462374" "GSM4462375"
[61] "GSM4462376" "GSM4462377" "GSM4462378" "GSM4462379" "GSM4462380"
[66] "GSM4462413" "GSM4462414" "GSM4462415" "GSM4462416"

[1] "X"          "GSM4432378" "GSM4432379" "GSM4432380" "GSM4432381"
 [6] "GSM4432382" "GSM4432383" "GSM4432384" "GSM4432385" "GSM4432386"
[11] "GSM4432387" "GSM4432388" "GSM4432389" "GSM4432390" "GSM4432391"
[16] "GSM4432392" "GSM4432393" "GSM4432394" "GSM4432395" "GSM4432396"
[21] "GSM4432397" "GSM4462336" "GSM4462337" "GSM4462338" "GSM4462339"
[26] "GSM4462340" "GSM4462341" "GSM4462342" "GSM4462343" "GSM4462344"
[31] "GSM4462345" "GSM4462346" "GSM4462347" "GSM4462348" "GSM4462349"
[36] "GSM4462350" "GSM4462351" "GSM4462352" "GSM4462353" "GSM4462354"
[41] "GSM4462355" "GSM4462356" "GSM4462357" "GSM4462358" "GSM4462359"
[46] "GSM4462360" "GSM4462361" "GSM4462362" "GSM4462363" "GSM4462364"
[51] "GSM4462365" "GSM4462366" "GSM4462367" "GSM4462368" "GSM4462369"
[56] "GSM4462370" "GSM4462371" "GSM4462372" "GSM4462373" "GSM4462374"
[61] "GSM4462375" "GSM4462376" "GSM4462377" "GSM4462378" "GSM4462379"
[66] "GSM4462380" "GSM4462413" "GSM4462414" "GSM4462415" "GSM4462416"

Run DESeq2 model

In [44]:
dds <- DESeqDataSetFromMatrix(countData=countDataFrameRound, colData=metaDataFrame, design=~treatment, tidy=FALSE)
dds

ERROR: Error in DESeqDataSetFromMatrix(countData = countDataFrameRound, colData = metaDataFrame, : ncol(countData) == nrow(colData) is not TRUE


Run DESEQ function for fitting and testin the model

In [15]:
dds <- DESeq(dds)

ERROR: Error in is(object, "DESeqDataSet"): object 'dds' not found


Take a look at the results table

In [13]:
res <- results(dds)
head(results(dds, tidy=TRUE))

ERROR: Error in is(object, "DESeqDataSet"): object 'dds' not found


Summary of differential gene expression

In [14]:
summary(res)

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'object' in selecting a method for function 'summary': object 'res' not found


Write DESeq2 data to file

In [15]:
write.csv(res, file = "GSE147507_results.csv")

ERROR: Error in is.data.frame(x): object 'res' not found


#Second Part: Results Analysis

Read results data

In [16]:
results <- read.csv("GSE147507_results.csv",header=T, sep=',')
head(results)

Warning message in file(file, "rt"):
"cannot open file 'GSE147507_results.csv': No such file or directory"


ERROR: Error in file(file, "rt"): cannot open the connection


Check null value

In [17]:
check_null <- is.na(results)
head(check_null)

Warning message in is.na(results):
"is.na() applied to non-(list or vector) of type 'closure'"


[1] FALSE

Omit the null value

In [18]:
results_omit_na <- na.omit(results)
head(results_omit_na)

                                                                                      
1 function (object, contrast, name, lfcThreshold = 0, altHypothesis = c("greaterAbs", 
2     "lessAbs", "greater", "less"), listValues = c(1, -1), cooksCutoff,              
3     independentFiltering = TRUE, alpha = 0.1, filter, theta,                        
4     pAdjustMethod = "BH", filterFun, format = c("DataFrame",                        
5         "GRanges", "GRangesList"), test, addMLE = FALSE, tidy = FALSE,              
6     parallel = FALSE, BPPARAM = bpparam(), minmu = 0.5)                             

Count the up regulated gene

In [19]:
results_omit_na_filter_up <- filter(results_omit_na, log2FoldChange>1 & padj<0.05)
head(results_omit_na_filter_up)
nrow(results_omit_na_filter_up)

ERROR: Error in UseMethod("filter"): no applicable method for 'filter' applied to an object of class "function"


Count the down regulated gene

In [20]:
results_omit_na_filter_down <- filter(results_omit_na, log2FoldChange<-1 & padj<0.05)
head(results_omit_na_filter_down)
nrow(results_omit_na_filter_down)

ERROR: Error in UseMethod("filter"): no applicable method for 'filter' applied to an object of class "function"


ABS logFC value and setup cuttoff criteria for p value

In [21]:
results_omit_na_filter <- filter(results_omit_na, abs(log2FoldChange)>1 & pvalue<0.05)
head(results_omit_na_filter)
nrow(results_omit_na_filter)

ERROR: Error in UseMethod("filter"): no applicable method for 'filter' applied to an object of class "function"


ABS logFC value and setup cuttoff criteria for p adj value

In [22]:
results_omit_na_filter <- filter(results_omit_na, abs(log2FoldChange)>1 & padj<0.05)
head(results_omit_na_filter)
nrow(results_omit_na_filter)

ERROR: Error in UseMethod("filter"): no applicable method for 'filter' applied to an object of class "function"


Write DESeq2 final results to file

In [23]:
write.csv(results_omit_na_filter, file="final_result_GSE147507.csv")

ERROR: Error in is.data.frame(x): object 'results_omit_na_filter' not found


#Third Part: Visualization (graphics)

Read final results

In [24]:
resultsShow<- read.csv("final_result_GSE147507.csv",header=T, sep=',')
head(resultsShow)

Warning message in file(file, "rt"):
"cannot open file 'final_result_GSE147507.csv': No such file or directory"


ERROR: Error in file(file, "rt"): cannot open the connection


#Sort summary list by p-value

In [25]:
res <- res[order(res$padj),]
head(res)

ERROR: Error in eval(expr, envir, enclos): object 'res' not found


#Plot Counts
We can use plotCounts function to compare the normalized counts between treated and control groups for our top 6 genes

In [26]:
par(mfrow=c(2,3))

plotCounts(dds, gene="TMEM204", intgroup="treatment")
plotCounts(dds, gene="SLC44A4", intgroup="treatment")
plotCounts(dds, gene="DUOX1", intgroup="treatment")
plotCounts(dds, gene="JAM2", intgroup="treatment")
plotCounts(dds, gene="CCL3", intgroup="treatment")
plotCounts(dds, gene="ADIRF", intgroup="treatment")

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'nrow': object 'dds' not found


Next steps in exploring these data...BLAST to database to find associated gene function

#Volcano Plot

In [27]:
#reset par
par(mfrow=c(1,1))
# Make a basic volcano plot
with(res, plot(log2FoldChange, -log10(pvalue), pch=20, main="Volcano plot", xlim=c(-3,3)))

# Add colored points: blue if padj<0.01, red if log2FC>1 and padj<0.05)
with(subset(res, padj<.05 ), points(log2FoldChange, -log10(pvalue), pch=20, col="blue"))
with(subset(res, padj<.05 & abs(log2FoldChange)>2), points(log2FoldChange, -log10(pvalue), pch=20, col="red"))

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'data' in selecting a method for function 'with': object 'res' not found


#PCA
First we need to transform the raw count data vst function will perform variance stabilizing transformation

Using the DESEQ2 plotPCA fxn we can

In [28]:
vsdata <- vst(dds, blind=FALSE)
plotPCA(vsdata, intgroup="treatment")

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'nrow': object 'dds' not found


Extract results for the top 250 up-regulated and top 250 down-regulated genes, sorted by p-value:

Print results for top and bottom 5 genes

In [29]:
n = 250 
resOrdered <- results_omit_na[order(results_omit_na$padj),]
topResults <- rbind( resOrdered[ resOrdered[,'log2FoldChange'] > 0, ][1:n,], resOrdered[ resOrdered[,'log2FoldChange'] < 0, ][n:1,] )
topResults[c(1:5,(2*n-4):(2*n)), c('X','baseMean','log2FoldChange','padj')]

ERROR: Error in results_omit_na$padj: object of type 'closure' is not subsettable


Plot counts for a single gene. Below is the plot for the gene with the lowest p-value:

In [30]:
plotCounts(dds, gene=which.min(results_omit_na$padj), intgroup='treatment', pch = 19)

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'which.min': object of type 'closure' is not subsettable
